# FNSPID preprocessing


In [1]:
%pip install -r requirements.txt

   ---------------------------------------- 0.0/13.2 MB ? eta -:--:--
   -- ------------------------------------- 0.8/13.2 MB 6.7 MB/s eta 0:00:02
   --------- ------------------------------ 3.1/13.2 MB 10.3 MB/s eta 0:00:01
   ----------- ---------------------------- 3.9/13.2 MB 10.7 MB/s eta 0:00:01
   -------------- ------------------------- 4.7/13.2 MB 6.5 MB/s eta 0:00:02
   --------------- ------------------------ 5.0/13.2 MB 5.5 MB/s eta 0:00:02
   --------------- ------------------------ 5.2/13.2 MB 4.7 MB/s eta 0:00:02
   ----------------- ---------------------- 5.8/13.2 MB 4.3 MB/s eta 0:00:02
   ------------------ --------------------- 6.0/13.2 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 8.1/13.2 MB 4.6 MB/s eta 0:00:02
   --------------------------- ------------ 9.2/13.2 MB 4.7 MB/s eta 0:00:01
   -------------------------------- ------- 10.7/13.2 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------  12.8/13.2 MB 5.4 MB/s eta 0:00:01
  


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import numpy as np
import pandas as pd
import preprocessing as pp
import math

MODE = "full"  

In [ ]:
info = pp.print_system_info()


System
------
CPU physical: 16
CPU logical: 24
RAM available: 4.45 GB
CUDA: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
VRAM free: 6.89 GB
Safe CPU threads: 8


In [ ]:
fnspid_dir = pp.find_fnspid_dir()
paths = pp.get_data_paths(fnspid_dir)
print("FNSPID:", fnspid_dir)
print("News file:", paths["news"])
print("News size GB:", round(paths["news"].stat().st_size / 1024**3, 2))
print("Price folder:", paths["price_dir"])

FNSPID: C:\Users\Admin\Downloads\algo_trading_fnspid\data\FNSPID
News file: C:\Users\Admin\Downloads\algo_trading_fnspid\data\FNSPID\Stock_news\nasdaq_exteral_data.csv
News size GB: 21.64
Price folder: C:\Users\Admin\Downloads\algo_trading_fnspid\data\FNSPID\Stock_price\full_history\full_history


In [ ]:
# Read CSV
news_sample = pp.inspect_news_schema(paths["news"], nrows=10)
news_columns = pp.detect_news_columns(paths["news"])


News CSV
--------
Columns: ['Unnamed: 0', 'Date', 'Article_title', 'Stock_symbol', 'Url', 'Publisher', 'Author', 'Article', 'Lsa_summary', 'Luhn_summary', 'Textrank_summary', 'Lexrank_summary']
Dtypes: {'Unnamed: 0': 'float64', 'Date': 'str', 'Article_title': 'str', 'Stock_symbol': 'str', 'Url': 'str', 'Publisher': 'float64', 'Author': 'float64', 'Article': 'str', 'Lsa_summary': 'str', 'Luhn_summary': 'str', 'Textrank_summary': 'str', 'Lexrank_summary': 'str'}
   Unnamed: 0                     Date  \
0         0.0  2023-12-16 23:00:00 UTC   
1         1.0  2023-12-12 00:00:00 UTC   
2         2.0  2023-12-12 00:00:00 UTC   
3         3.0  2023-12-07 00:00:00 UTC   
4         4.0  2023-12-07 00:00:00 UTC   

                                       Article_title Stock_symbol  \
0  Interesting A Put And Call Options For August ...            A   
1  Wolfe Research Initiates Coverage of Agilent T...            A   
2  Agilent Technologies Reaches Analyst Target Price            A   
3  Ag

In [ ]:
# Select the company universe and the training cutoff date
universe, universe_train_end = pp.select_company_universe(fnspid_dir, mode=MODE)
print("Universe train cutoff:", universe_train_end)
display(universe.head(30))


Detected news columns
---------------------
index: Unnamed: 0
date: Date
title: Article_title
symbol: Stock_symbol
url: Url
publisher: Publisher
author: Author
article: Article
lsa_summary: Lsa_summary
luhn_summary: Luhn_summary
textrank_summary: Textrank_summary
lexrank_summary: Lexrank_summary
News start: 1999-08-31 00:00:00
News end: 2024-01-09 00:00:00
Universe selection uses news only through: 2016-09-17

Universe filter audit
---------------------
News-qualified candidates checked: 5,122
Rejected - missing price file: 6
Rejected - < 250 train price days: 1
Rejected - unreadable price file: 0
Accepted: 50 / target 50

Company universe
----------------
Companies: 50
ticker  train_news  train_price_days         price_start           price_end
   QQQ        4855              4290 1999-03-10 00:00:00 2023-12-28 00:00:00
   GLD        4482              2978 2004-11-18 00:00:00 2023-12-28 00:00:00
  GILD        4367              4290 1992-01-22 00:00:00 2023-12-28 00:00:00
  DISH      

,ticker,train_news,train_price_days,price_start,price_end
0,QQQ,4855,4290,1999-03-10 00:00:00,2023-12-28 00:00:00
1,GLD,4482,2978,2004-11-18 00:00:00,2023-12-28 00:00:00
2,GILD,4367,4290,1992-01-22 00:00:00,2023-12-28 00:00:00
3,DISH,4331,4290,1995-06-21 00:00:00,2023-12-28 00:00:00
4,KO,4170,4291,1962-01-02 00:00:00,2023-12-28 00:00:00
5,GRPN,4036,1224,2011-11-04 00:00:00,2023-12-28 00:00:00
6,MRK,3565,4290,1970-01-02 00:00:00,2023-12-28 00:00:00
7,EBAY,3535,4290,1998-09-24 00:00:00,2023-12-28 00:00:00
8,ORCL,3530,4290,1986-03-12 00:00:00,2023-12-28 00:00:00
9,NOK,3430,4290,1994-07-01 00:00:00,2023-12-28 00:00:00


In [7]:
# Check one stock CSV
first_ticker = universe.iloc[0]["ticker"]
price_file = paths["price_dir"] / f"{first_ticker}.csv"
pp.inspect_price_schema(price_file, nrows=10)


Price CSV
---------
Columns: ['date', 'volume', 'open', 'high', 'low', 'close', 'adj close']
Dtypes: {'date': 'str', 'volume': 'int64', 'open': 'float64', 'high': 'float64', 'low': 'float64', 'close': 'float64', 'adj close': 'float64'}
         date    volume        open        high         low       close  \
0  2023-12-28  26956300  412.670013  412.920013  410.940002  411.299988   
1  2023-12-27  31980500  410.950012  411.790009  410.079987  411.500000   
2  2023-12-26  22722500  409.250000  411.559998  409.149994  410.880005   
3  2023-12-22  34292400  409.000000  409.970001  406.480011  408.380005   
4  2023-12-21  45568900  407.059998  408.140015  404.470001  407.769989   

    adj close  
0  411.299988  
1  411.500000  
2  410.664001  
3  408.165314  
4  407.555634  


,date,volume,open,high,low,close,adj close
0,2023-12-28,26956300,412.670013,412.920013,410.940002,411.299988,411.299988
1,2023-12-27,31980500,410.950012,411.790009,410.079987,411.500000,411.500000
2,2023-12-26,22722500,409.250000,411.559998,409.149994,410.880005,410.664001
3,2023-12-22,34292400,409.000000,409.970001,406.480011,408.380005,408.165314
4,2023-12-21,45568900,407.059998,408.140015,404.470001,407.769989,407.555634
5,2023-12-20,54042400,408.350006,410.470001,402.899994,403.079987,402.868073
6,2023-12-19,35711900,407.540009,409.279999,407.350006,409.160004,408.944916
7,2023-12-18,46610000,404.929993,407.989990,404.600006,407.079987,406.865967
8,2023-12-15,62598000,404.179993,406.540009,403.570007,405.339996,404.319305
9,2023-12-14,55447800,404.980011,406.299988,400.339996,403.390015,402.374237


In [8]:
# Extract news for the selected companies
news = pp.extract_panel_news(fnspid_dir, mode=MODE)
print("News rows:", len(news))
print("Companies in news:", news["ticker"].nunique())
print("Rows per company:")
display(news.groupby("ticker").size().sort_values(ascending=False).head(20).rename("articles"))


Detected news columns
---------------------
index: Unnamed: 0
date: Date
title: Article_title
symbol: Stock_symbol
url: Url
publisher: Publisher
author: Author
article: Article
lsa_summary: Lsa_summary
luhn_summary: Luhn_summary
textrank_summary: Textrank_summary
lexrank_summary: Lexrank_summary

Detected news columns
---------------------
index: Unnamed: 0
date: Date
title: Article_title
symbol: Stock_symbol
url: Url
publisher: Publisher
author: Author
article: Article
lsa_summary: Lsa_summary
luhn_summary: Luhn_summary
textrank_summary: Textrank_summary
lexrank_summary: Lexrank_summary
Using cached company universe.

Panel news
----------
Rows before cleaning: 348,866
Rows after cleaning: 348,410
Rows removed: 456
Companies: 50
Extraction method: DuckDB
Saved: C:\Users\Admin\Downloads\algo_trading_fnspid\data\cache\panel_news.pkl
Extraction time: 40.62 sec
News rows: 348410
Companies in news: 50
Rows per company:


ticker
GILD     12362
QQQ      11813
WFC      11301
MRK      10774
KO       10520
MU        9605
MS        9458
QCOM      8954
CMCSA     8820
FDX       8731
GLD       8338
ORCL      7998
AMT       7900
CAT       7514
SLB       7372
EBAY      7178
CMG       7174
GME       7130
PEP       7113
EA        7085
Name: articles, dtype: int64

In [9]:
# Check timestamp quality
timestamp_info = pp.audit_timestamp_quality(news)
print(timestamp_info)


Timestamp audit
---------------
Rows: 348,410
Parsed: 348,410
Unparsed: 0
Date-only ratio: 99.83%
UTC hour counts for precise timestamps:
published_raw
0       5
1      15
2      13
3      28
4      54
5      69
6     145
7      58
8      45
9      27
10     23
11     15
12     24
13     11
14      5
15      5
16      3
17      4
18      2
19      3
20     10
21      2
22      9
23      8
{'rows': 348410, 'parsed': 348410, 'unparsed': 0, 'date_only_ratio': 0.9983266840791022, 'distinct_precise_hours': 24}


In [10]:
# Load prices
prices = pp.load_panel_prices(fnspid_dir, mode=MODE)
print("Price rows:", len(prices))
print("Companies:", prices["ticker"].nunique())
print("Date range:", prices["date"].min(), "to", prices["date"].max())
price_info = prices.groupby("ticker").agg(rows=("date", "size"), start=("date", "min"), end=("date", "max"))
display(price_info.head(20))


Detected news columns
---------------------
index: Unnamed: 0
date: Date
title: Article_title
symbol: Stock_symbol
url: Url
publisher: Publisher
author: Author
article: Article
lsa_summary: Lsa_summary
luhn_summary: Luhn_summary
textrank_summary: Textrank_summary
lexrank_summary: Lexrank_summary
Using cached company universe.


Load prices: 100%|██████████| 50/50 [00:01<00:00, 25.64it/s]


Price rows: 424859
Companies: 50
Date range: 1962-01-02 00:00:00 to 2023-12-28 00:00:00


,rows,start,end
ticker,,,
AA,15605,1962-01-02,2023-12-28
AEO,7480,1994-04-14,2023-12-28
AIG,12860,1973-01-02,2023-12-28
AMT,6502,1998-02-27,2023-12-28
BIIB,8132,1991-09-17,2023-12-28
BSX,7962,1992-05-19,2023-12-28
CAT,13618,1970-01-02,2023-12-28
CI,10525,1982-03-31,2023-12-28
CMCSA,11040,1980-03-17,2023-12-28


In [11]:
# Run FinBERT
sentiment = pp.run_finbert(news, prices, mode=MODE)
print("Sentiment rows:", len(sentiment))
print("Feature date range:", sentiment["feature_date"].min(), "to", sentiment["feature_date"].max())
display(sentiment[["ticker", "feature_date", "title", "sentiment_score"]].head())

Align news dates: 100%|██████████| 50/50 [03:46<00:00,  4.52s/it]
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\Downloads\algo_trading_fnspid\data\FNSPID\.cache\huggingface\models--ProsusAI--finbert. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
T

FinBERT labels: {0: 'positive', 1: 'negative', 2: 'neutral'}
FinBERT device: cuda
Tokenizer max length: 56
Batch 16: 191.1 articles/sec
Batch 32: 981.6 articles/sec
Batch 64: 1476.3 articles/sec
Batch 128: 2462.5 articles/sec
FinBERT batch size: 128


FinBERT: 100%|██████████| 341282/341282 [02:49<00:00, 2014.83it/s]


FinBERT sanity check: {'positive': 0.9463821053504944, 'negative': 0.028025511652231216, 'neutral': 0.025592386722564697}
Saved: C:\Users\Admin\Downloads\algo_trading_fnspid\data\cache\panel_news_sentiment.pkl
FinBERT time: 169.38 sec
FinBERT speed: 2014.8 articles/sec
Sentiment rows: 341282
Feature date range: 2009-04-15 00:00:00 to 2023-12-18 00:00:00


,ticker,feature_date,title,sentiment_score
0,AA,2009-08-11,Lithium One Reports Initial Drill Results for ...,0.448864
1,AA,2009-08-11,"Breaking News: China Armco Metals - August 10,...",-0.016755
2,AA,2009-08-11,Andean Resources Ltd.: Step-Out Drilling Exten...,0.809810
3,AA,2009-08-11,PC Gold Opens Up Pickle Crow With Discovery of...,0.253893
4,AA,2009-08-11,Ahead of the Bell: China Armco Metals - August...,-0.001298


In [12]:
# Daily sentiment
daily = pp.build_daily_sentiment(sentiment, mode=MODE)
print("Daily rows:", len(daily))
print("Trading days with news:", daily["date"].nunique())
display(daily.head())

Saved: C:\Users\Admin\Downloads\algo_trading_fnspid\data\cache\panel_daily_sentiment.pkl
Daily rows: 111460
Trading days with news: 3615


,ticker,date,sentiment_sum,sentiment_pos_sum,sentiment_neg_sum,sentiment_neu_sum,sentiment_mean,sentiment_std,news_count
0,AA,2009-08-11,1.493215,1.644588,0.151373,4.204038,0.248869,0.331783,6
1,AA,2009-09-04,-0.964569,0.009616,0.974185,0.016199,-0.964569,0.000000,1
2,AA,2009-09-21,-0.861411,0.018385,0.879796,0.101819,-0.861411,0.000000,1
3,AA,2009-09-22,0.058217,0.072741,0.014524,0.912735,0.058217,0.000000,1
4,AA,2009-09-24,0.036131,0.051409,0.015278,0.933313,0.036131,0.000000,1


In [13]:
# Build the panel dataset
panel = pp.build_panel_base(prices, daily, mode=MODE)
print("Panel rows:", len(panel))
print("Companies:", panel["ticker"].nunique())
print("Date range:", panel["date"].min(), "to", panel["date"].max())
print("Labeled rows:", panel["target_return"].notna().sum())
print("Future rows:", panel["target_return"].isna().sum())
display(panel.head())

Saved: C:\Users\Admin\Downloads\algo_trading_fnspid\data\cache\panel_base.pkl
Panel rows: 424,859
Panel companies: 50
Panel rows: 424859
Companies: 50
Date range: 1962-01-02 00:00:00 to 2023-12-28 00:00:00
Labeled rows: 424809
Future rows: 50


,date,open,high,low,close,volume,adj_close,ticker,prev_close,prev_volume,...,target_open,target_date,target_return,sentiment_sum,sentiment_pos_sum,sentiment_neg_sum,sentiment_neu_sum,sentiment_mean,sentiment_std,news_count
0,1962-01-02,6.532155,6.556185,6.532155,6.532155,55900.0,1.536658,AA,NaN,NaN,...,6.532155,1962-01-03,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1962-01-03,6.532155,6.632280,6.524145,6.632280,74500.0,1.560212,AA,6.532155,55900.0,...,6.632280,1962-01-04,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1962-01-04,6.632280,6.664320,6.632280,6.632280,80500.0,1.560212,AA,6.632280,74500.0,...,6.632280,1962-01-05,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1962-01-05,6.632280,6.656310,6.616260,6.624270,70500.0,1.558326,AA,6.632280,80500.0,...,6.608250,1962-01-08,-0.002418,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1962-01-08,6.608250,6.608250,6.339915,6.408000,93800.0,1.507450,AA,6.624270,70500.0,...,6.408000,1962-01-09,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
# Check lookback features
check = pp.make_lookback_features(panel, lookback=7, min_history=pp.REQUIRED_HISTORY)
features = pp.model_feature_columns()
print("Model features:", features)
print("Rows after common warm-up:", len(check))
print("Missing feature values:")
print(check[features].isna().sum())

Model features: ['daily_return', 'open_gap', 'intraday_return', 'high_low_range', 'volume_change', 'sentiment_mean', 'sentiment_std', 'news_count', 'return_mean_lb', 'return_vol_lb', 'volume_change_mean_lb', 'sentiment_mean_lb', 'sentiment_std_lb', 'news_count_lb']
Rows after common warm-up: 423359
Missing feature values:
daily_return                 0
open_gap                     0
intraday_return          11578
high_low_range               0
volume_change             1865
sentiment_mean               0
sentiment_std                0
news_count                   0
return_mean_lb               0
return_vol_lb                0
volume_change_mean_lb     2250
sentiment_mean_lb            0
sentiment_std_lb             0
news_count_lb                0
dtype: int64


In [15]:
# Final preprocessing checks and summary
panel_sorted = panel.sort_values(["ticker", "date"]).reset_index(drop=True)
group = panel_sorted.groupby("ticker", group_keys=False)
expected_prev_close = group["close"].shift(1)
expected_target_open = group["open"].shift(-1)
expected_target_date = group["date"].shift(-1)

assert np.allclose(
    panel_sorted["prev_close"].fillna(0).to_numpy(),
    expected_prev_close.fillna(0).to_numpy(),
)
assert np.allclose(
    panel_sorted["target_open"].fillna(0).to_numpy(),
    expected_target_open.fillna(0).to_numpy(),
)
assert panel_sorted["target_date"].fillna(pd.Timestamp("1900-01-01")).equals(
    expected_target_date.fillna(pd.Timestamp("1900-01-01"))
)
assert (panel_sorted.groupby("ticker", sort=False).head(1)["history_count"] == 1).all()
assert (panel_sorted.dropna(subset=["target_date"])["target_date"] > panel_sorted.dropna(subset=["target_date"])["date"]).all()
for ticker, group in panel_sorted.groupby("ticker"):
    assert group["date"].is_monotonic_increasing

summary = {
    "mode": MODE,
    "companies": int(panel["ticker"].nunique()),
    "rows": int(len(panel)),
    "date_start": str(panel["date"].min()),
    "date_end": str(panel["date"].max()),
    "news_rows": int(len(news)),
    "universe_train_end": str(universe_train_end),
    "timestamp_audit": timestamp_info,
}
pp.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
with open(pp.OUTPUT_DIR / "preprocessing_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Preprocessing checks: PASS")
print("Saved preprocessing_summary.json")


Preprocessing checks: PASS
Saved preprocessing_summary.json
